In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("../../").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(
        str(PROJECT_ROOT)
    )

print(PROJECT_ROOT)

/home/ubuntu/Projects/thesis-code


In [5]:
from src.baseline.dataset import COdeBaselineDataset
from src.baseline.transforms import get_image_transform
from src.baseline import config


dataset = COdeBaselineDataset(
    csv_path=config.DATASET_PATH,
    split="train",
    image_root=config.IMAGE_ROOT,
    transform=get_image_transform(),
)


sample = dataset[0]


print(sample["checkup_id"])
print(len(sample["images"]))
print(sample["images"][0].shape)
print(sample["labels"].shape)

train: 6129 samples
0001-001
1
torch.Size([3, 224, 224])
torch.Size([13])


In [6]:
# TEMPORARY SANITY CHECK — Remove after successful validation
# Purpose:
# Validate Radiograph-only dataset filtering and modality availability

from src.baseline.dataset import COdeBaselineDataset
from src.baseline import config
from src.baseline.transforms import get_image_transform


dataset = COdeBaselineDataset(
    csv_path=config.DATASET_PATH,
    split="train",
    image_root=config.IMAGE_ROOT,
    transform=get_image_transform(),
    require_modality=config.REQUIRE_MODALITY,
)


print("\nDataset length:", len(dataset))


sample = dataset[0]


print("\nSample keys:")
print(sample.keys())


print("\nCheckup ID:")
print(sample["checkup_id"])


print("\nPatient ID:")
print(sample["patient_id"])


print("\nPhotograph count:")
print(len(sample["images"]))


print("\nRadiograph count:")
print(len(sample["radiographs"]))


print("\nLabel shape:")
print(sample["labels"].shape)


empty_radiograph_count = 0


for i in range(len(dataset)):

    if len(dataset[i]["radiographs"]) == 0:
        empty_radiograph_count += 1


print("\nEmpty radiograph samples:")
print(empty_radiograph_count)

train: 2972 samples
Required modality: radiograph

Dataset length: 2972

Sample keys:
dict_keys(['checkup_id', 'patient_id', 'images', 'radiographs', 'text', 'labels'])

Checkup ID:
0001-001

Patient ID:
1

Photograph count:
1

Radiograph count:
1

Label shape:
torch.Size([13])

Empty radiograph samples:
0


In [7]:
# TEMPORARY SANITY CHECK — Remove after successful validation
# Purpose:
# Validate Radiograph-only DataLoader batch structure

from torch.utils.data import DataLoader

from src.baseline.collate import baseline_collate


loader = DataLoader(
    dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    collate_fn=baseline_collate,
)


batch = next(iter(loader))


print("Batch keys:")
print(batch.keys())


print("\nNumber of samples:")
print(len(batch["radiographs"]))


print("\nFirst sample radiograph count:")
print(len(batch["radiographs"][0]))


print("\nFirst radiograph tensor shape:")
print(batch["radiographs"][0][0].shape)


print("\nLabels shape:")
print(batch["labels"].shape)

Batch keys:
dict_keys(['checkup_id', 'patient_id', 'images', 'radiographs', 'text', 'labels'])

Number of samples:
16

First sample radiograph count:
2

First radiograph tensor shape:
torch.Size([3, 224, 224])

Labels shape:
torch.Size([16, 13])


# Thesis Note 05.1
# Baseline Experiment A — Radiograph-only Fine-tuned Classification

## Overview

This experiment represents the first official unimodal baseline for the COde dataset classification task.

The objective is to evaluate how much diagnostic information can be extracted from **radiographic images alone** before introducing additional modalities such as photographs and clinical text.

This baseline establishes a reference point for future experiments including:

- Photograph-only models
- Text-only models
- Multimodal fusion models
- Missing-modality robust models

---

# 1. Experiment Objective

The goal of this experiment is to train and evaluate a radiograph-only deep learning classifier for multi-label dental diagnosis prediction.

The model receives only radiographic images from each patient visit and predicts the presence of 13 reconstructed dental conditions.

The experiment answers the following research question:

> How effective are radiographic images alone for automated multi-label dental diagnosis classification on the COde dataset?

---

# 2. Dataset Configuration

## Dataset

Dataset:
COde Dataset


Task:
Multi-label dental diagnosis classification


Number of labels:
13 labels


The labels were reconstructed from clinical information and attached to the patient-level split dataset.

---

## Patient-Level Split

The experiment uses the previously validated patient-level split.

Split strategy:
Patient-level split

Seed:
42


This ensures that samples from the same patient cannot appear in multiple partitions.

Leakage checks previously confirmed:
Patient overlap: 0
Visit overlap: 0
Image overlap: 0


Therefore, this experiment follows a leakage-safe evaluation protocol.

---

# 3. Input Modality

This experiment uses only radiographic images.

Used modality:
Radiographs


Excluded modalities:
Photographs
Clinical text


The input pipeline:

|
v
ResNet50 Encoder
|
v
Feature Aggregation
|
v
Classification Head
|
v
13-label Prediction


---

# 4. Model Architecture

## Image Encoder

Backbone:
ResNet50

Initialization:
ImageNet pretrained weights


Training strategy:
Encoder fine-tuning


Unlike the initial frozen encoder baseline, the feature extractor parameters are updated during training.

---

## Variable-Length Radiograph Aggregation

Each dental visit may contain a different number of radiographs.

To handle this variable length input, each radiograph is independently encoded and the extracted features are aggregated using mean pooling.

Architecture:
Radiograph 1 ----
Radiograph 2 ----- ResNet50 ---- Mean Pooling ---- Classifier
Radiograph N ----/


Mathematically:

\[
z = \frac{1}{N}\sum_{i=1}^{N}f(x_i)
\]

where:

- \(x_i\) represents the i-th radiograph
- \(f(.)\) represents the ResNet50 encoder
- \(z\) is the visit-level representation

---

# 5. Training Configuration

Training samples:
2972

Validation samples:
642

Number of epochs:
20

Loss function:
Binary Cross Entropy with Logits Loss


Task formulation:
Multi-label classification


---

## Best Checkpoint

The best model checkpoint was obtained at:
Epoch 19


based on validation performance.

Saved checkpoint:
results/baseline/radiograph_only_finetune/best_model.pt


---

# 6. Threshold Optimization

Because the task is multi-label classification and each disease has different prevalence, using a fixed threshold of 0.5 is not optimal.

Therefore, thresholds were optimized separately for each label.

Procedure:

1. Train the model using the training split.
2. Generate predictions on validation data.
3. Search the optimal threshold for each label based on F1-score.
4. Apply the learned thresholds to test evaluation.

Important:

The test split was never used during threshold selection.

---

# 7. Validation Results

Evaluation split:
Validation


Number of samples:
1330 visits

## Default Threshold (0.5)

| Metric | Score |
|---|---:|
| Macro F1 | 0.1641 |
| Micro F1 | 0.2614 |
| AUROC | 0.5782 |

---

## Optimized Thresholds

| Metric | Score |
|---|---:|
| Macro F1 | 0.2481 |
| Micro F1 | 0.2206 |
| AUROC | 0.5782 |

---

# 8. Test Results

Evaluation split:
Test

Number of samples:
1316 visits

Thresholds:
Learned from validation split

---

## Default Threshold (0.5)

| Metric | Score |
|---|---:|
| Macro F1 | 0.1695 |
| Micro F1 | 0.2550 |
| AUROC | 0.5894 |

---

## Optimized Thresholds

| Metric | Score |
|---|---:|
| Macro F1 | 0.2401 |
| Micro F1 | 0.2198 |
| AUROC | 0.5894 |

---

# 9. Observations

## Effect of Encoder Fine-tuning

Compared with the frozen encoder baseline:

- Feature fine-tuning improved classification performance.
- The model learned dataset-specific dental visual representations.
- Validation AUROC increased to approximately 0.70 during training.

---

## Effect of Threshold Optimization

Threshold optimization significantly improved Macro F1:

Validation:
0.1641 → 0.2481

Test:
0.1695 → 0.2401

This confirms that label-wise thresholds are important for imbalanced multi-label dental diagnosis tasks.

---

## Generalization

The test AUROC:
0.5894

was slightly higher than validation:
0.5782


indicating that the model generalizes reasonably under the patient-level split.

---

# 10. Limitations

This baseline has several limitations:

- Radiographs are naturally missing for many visits in the COde dataset.
- Mean pooling ignores spatial relationships between multiple radiographs.
- Label reconstruction introduces noise due to weak supervision.
- Only one modality is utilized.

---

# 11. Role in Thesis

This experiment establishes the first official visual baseline.

The results provide a reference point for evaluating future approaches:

1. Photograph-only baseline
2. Text-only baseline
3. Multimodal fusion
4. Missing-modality learning methods

The main purpose of this experiment is not to achieve the final best performance, but to quantify the contribution of radiographic information alone.
